# 07 · Evaluación offline: SmolVLA contra la primera ACT

La misma vara que `05`: en los 7 episodios que ninguno de los dos vio al
entrenar, en distintos instantes, cada política recibe la observación grabada
y predice su bloque; lo comparamos con lo que hizo la persona. Las dos
referencias tontas de siempre dan la escala: quedarse quieto, y repetir la
última acción.

Ojo con lo que esto mide: el modelo siempre recibe la observación verdadera,
nunca la consecuencia de sus propias acciones. Dice si aprendió, no si la
tarea saldría en el robot.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.eval.offline import evaluate_episode
from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy.load import LoadedPolicy, load_policy
from toolkit.policy.smolvla_g1_base import load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"

ds = LeRobotDataset("local/pick_mameluco_act_v1", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1")
split = json.loads((REPO_ROOT / "notes" / "act-split.json").read_text())
names = ds.meta.features["action"]["names"]
JOINTS = names[0] if isinstance(names[0], list) else names

act = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device=device)
train_meta = LeRobotDataset("local/pick_mameluco_act_v1_train", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1_train").meta
smol_policy, smol_pre, smol_post = load_g1_policy(
    CHECKPOINTS_DIR / "smolvla_g1_v1" / "checkpoints" / "last" / "pretrained_model", train_meta.stats, device)
smol = LoadedPolicy(smol_policy, smol_pre, smol_post, [])
print("bloque · ACT:", act.policy.config.chunk_size, "· SmolVLA:", smol.policy.config.chunk_size)

Los bloques no tienen el mismo largo (ACT 48, SmolVLA 50). Para comparar
igual, evaluamos a los dos en los **mismos instantes** (cada 48 cuadros) y
medimos sólo las primeras 48 acciones de cada bloque.

In [ ]:
HORIZON = 48
torch.manual_seed(0)
predictions = {"ACT": [], "SmolVLA": []}
for episode in split["heldout_episode_indices"]:
    predictions["ACT"] += evaluate_episode(act, ds, episode, stride=HORIZON)
    predictions["SmolVLA"] += evaluate_episode(smol, ds, episode, stride=HORIZON)

common = {(p.episode_index, p.start) for p in predictions["ACT"]} & {(p.episode_index, p.start) for p in predictions["SmolVLA"]}
by_key = {name: {(p.episode_index, p.start): p for p in preds if (p.episode_index, p.start) in common}
          for name, preds in predictions.items()}
keys = sorted(common)
print(f"{len(keys)} bloques en {len(split['heldout_episode_indices'])} episodios no vistos")


def errors(name, field):
    """Error absoluto (rad) por bloque, paso y articulación: (bloques, 48, 26)."""
    return np.stack([np.abs(getattr(by_key[name][k], field)[:HORIZON] - by_key[name][k].target[:HORIZON]) for k in keys])


table = {
    "ACT": errors("ACT", "predicted"),
    "SmolVLA": errors("SmolVLA", "predicted"),
    "quedarse quieto": errors("ACT", "hold"),
    "repetir la última": errors("ACT", "last_action"),
}
summary = pd.DataFrame({name: {"error medio (rad)": e.mean(), "error medio (°)": np.degrees(e.mean())} for name, e in table.items()}).T
summary.round(4)

**Empate.** SmolVLA se equivoca 2,58° en promedio y ACT 2,67°. Es una
diferencia de 3 %, con un solo sorteo y 52 bloques: no alcanza para decir que
uno sea mejor. Los dos están muy lejos de las referencias tontas (quedarse
quieto 6,38°, repetir la última acción 3,85°), así que los dos aprendieron algo
de verdad.

Con el punto de control del paso 5000, SmolVLA daba 2,65°. Los 15.000 pasos
siguientes bajaron la pérdida de entrenamiento 2,5 veces (de 0,039 a 0,016),
pero el error en episodios no vistos sólo un 3 %. La pérdida de entrenamiento
dejó de ser una buena guía de lo que pasa con episodios nuevos.

## Según cuán lejos predice

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
styles = {"ACT": ("tab:blue", "-"), "SmolVLA": ("tab:red", "-"), "quedarse quieto": ("gray", "--"), "repetir la última": ("gray", ":")}
for name, e in table.items():
    color, style = styles[name]
    ax.plot(np.arange(HORIZON) / 30, np.degrees(e.mean(axis=(0, 2))), color=color, ls=style, label=name)
ax.set_xlabel("segundos hacia adelante")
ax.set_ylabel("error medio (°)")
ax.legend()
plt.tight_layout()

Las dos curvas casi se superponen: arrancan entre 1,4° y 1,7° y llegan a
3,4° a los 1,6 s. Predecir más lejos es igual de difícil para las dos.

Lo llamativo es el principio: durante los primeros 0,3 s, **repetir la última
acción les gana a las dos políticas**. En el primer cuadro esa referencia casi
no se equivoca (el brazo todavía no se movió), mientras que las políticas ya
arrancan corridas. Es el salto al empezar cada bloque que vimos en la primera
ACT (`notes/act-first-model.md`): en el robot se nota como un tirón cada vez
que cambia el bloque. SmolVLA salta un poco menos (1,4° contra 1,65°).

## Por articulación

In [ ]:
per_joint = pd.DataFrame({name: np.degrees(table[name].mean(axis=(0, 1))) for name in ["ACT", "SmolVLA", "quedarse quieto"]},
                         index=JOINTS).sort_values("SmolVLA")
ax = per_joint.plot.barh(figsize=(8, 8), color=["tab:blue", "tab:red", "lightgray"])
ax.set_xlabel("error medio (°)")
plt.tight_layout()
better = (per_joint["SmolVLA"] < per_joint["ACT"]).sum()
print(f"SmolVLA tiene menos error que ACT en {better} de 26 articulaciones")

SmolVLA tiene menos error en 15 de 26 articulaciones (en el paso 5000 eran
13). Donde más se separan:

- **Codo derecho y muñecas**: gana SmolVLA. En el codo derecho, 4,6° contra
  5,3°; en la inclinación de las muñecas, casi 1° menos.
- **Dedos de la mano derecha**: gana ACT, por poco (entre 0,1° y 0,4°).

Los codos son lo que más error tiene en las dos, porque son lo que más se
mueve: quedarse quieto se equivoca entre 11° y 12° ahí.

Y una alerta: en la inclinación de las dos muñecas (subir o bajar la mano),
que casi no se mueve, **las dos políticas se equivocan más que quedarse
quieto**. Agregan un movimiento que la persona no hizo, igual que la muñeca
que subía en L7.6.

## Un episodio entero

El codo derecho en un episodio no visto: en gris lo que hizo la persona, y los
bloques de cada política desde cada instante.

In [ ]:
episode = split["heldout_episode_indices"][1]
joint = JOINTS.index("kRightElbow")
rows = ds.meta.episodes
start, stop = int(rows["dataset_from_index"][episode]), int(rows["dataset_to_index"][episode])
truth = np.stack(ds.hf_dataset.with_format("numpy")[start:stop]["action"])

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, name, color in [(axes[0], "ACT", "tab:blue"), (axes[1], "SmolVLA", "tab:red")]:
    ax.plot(truth[:, joint], color="0.6", lw=3, label="la persona")
    for (e, s), p in by_key[name].items():
        if e == episode:
            ax.plot(range(s, s + HORIZON), p.predicted[:HORIZON, joint], color=color, lw=1.2)
    ax.set_title(f"{name} · episodio {episode} · {JOINTS[joint]}", fontsize=10)
axes[1].set_xlabel("cuadro")
axes[0].legend(fontsize=8)
plt.tight_layout()

## Cuánto tarda

En la misma máquina, un bloque de cada una:

In [ ]:
frame = ds[int(rows["dataset_from_index"][episode]) + 100]


def timed(loaded, repeats=10):
    from toolkit.eval.offline import predict_chunk
    predict_chunk(loaded, frame)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        predict_chunk(loaded, frame)
    if device == "cuda":
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / repeats


print(f"ACT: {timed(act):.0f} ms por bloque · SmolVLA: {timed(smol):.0f} ms por bloque ({device})")

ACT tarda 5 ms y SmolVLA entre 130 y 150 ms (varía entre corridas): unas 25 a
30 veces más. Según L7.6, unos 20 a 30 ms son la fila del VLM, que se calcula una vez, y el resto son las 10 pasadas del
experto.

¿Importa? Un bloque son 50 acciones a 30 por segundo: 1,7 s de movimiento. Si
el robot ejecuta un bloque mientras se calcula el siguiente, sobra tiempo.
Lo que SmolVLA no puede es recalcular en cada cuadro (cada 33 ms), que es lo
que necesita el promedio de bloques ("temporal ensembling") que suavizaba a la
primera ACT. El paper propone para eso la inferencia asíncrona: calcular el
próximo bloque mientras se ejecuta el actual.

Estos tiempos son de la GPU de esta workstation. En la computadora del robot
no lo medimos.

## Lo que nos llevamos

- En 7 episodios que nunca vio, SmolVLA y ACT empatan: 2,58° contra 2,67°.
  Las dos aprendieron de verdad (quedarse quieto da 6,38°).
- SmolVLA anda mejor en el codo derecho y las muñecas, y ACT en los dedos de
  la mano derecha. Las dos inventan movimiento en la inclinación de las
  muñecas.
- Las dos arrancan cada bloque corridas un grado y medio: el salto al cambiar
  de bloque sigue ahí.
- Pasar de 5000 a 20.000 pasos bajó la pérdida 2,5 veces y el error en
  episodios nuevos apenas un 3 %.
- SmolVLA tarda unas 25 a 30 veces más (130 a 150 ms contra 5 ms). Alcanza para un bloque de
  1,7 s, pero no para recalcular en cada cuadro.
- Nuestra lectura, sin verificar: lo que SmolVLA trae de más (leer la
  instrucción, haber visto muchos robots) no tiene dónde lucirse acá. Hay una
  sola tarea, una sola instrucción y una sola cámara.

## Para que la comparación sea justa (lo que no es igual)

- **Entrenamiento distinto.** La primera ACT se entrenó fuera de este repo (30.000
  pasos, con la fuga de normalización de `notes/act-first-model.md`). SmolVLA,
  20.000 pasos con lote 32 y estadísticas sólo de entrenamiento.
- **Un solo sorteo.** SmolVLA sortea; acá medimos un sorteo por instante.
- **Sin lazo cerrado.** Lo que importa al final es si agarra el mameluco en el
  robot, y eso esta evaluación no lo dice.